[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hisaylama/Brownian-Simulation/blob/master/Langevin_simulation.ipynb)

# Active Brownian Motion — Full Langevin Simulation

Simulates an **active Brownian particle** (microswimmer) in a homogeneous 2-D environment using the overdamped Langevin equations:

$$\dot{x} = v\cos\theta + \sqrt{2D_T}\,\xi_x(t)$$
$$\dot{y} = v\sin\theta + \sqrt{2D_T}\,\xi_y(t)$$
$$\dot{\theta} = \Omega + \sqrt{2D_R}\,\xi_\theta(t)$$

where $\xi$ are independent Gaussian white-noise terms, $D_T$ is the **translational** diffusion coefficient, $D_R$ is the **rotational** diffusion coefficient, $v$ is the self-propulsion speed, and $\Omega$ is the angular velocity (chirality).

Reference: Volpe et al., *Simulation of the active Brownian motion of microswimmers*, Am. J. Phys. **82**, 659 (2014).

## 1 — Physical parameters and diffusion coefficients

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── physical constants ────────────────────────────────────────────────────────
kB  = 1.38e-23   # Boltzmann constant  [J/K]
T   = 300        # temperature         [K]
eta = 0.001      # dynamic viscosity   [Pa·s]  (water at ~20 °C)
R   = 1e-6       # particle radius     [m]     (1 µm)

# ── diffusion coefficients (Stokes–Einstein) ──────────────────────────────────
D_T = kB * T / (6 * np.pi * eta * R)      # translational  [m²/s]
D_R = kB * T / (8 * np.pi * eta * R**3)   # rotational     [rad²/s]

# ── simulation parameters ─────────────────────────────────────────────────────
dt   = 1e-3      # time step           [s]
N    = 10_000    # number of steps
v    = 1e-5      # self-propulsion speed [m/s]
W    = np.pi     # angular velocity    [rad/s]  (one revolution per ~2 s)

print(f"Translational diffusion coefficient  D_T = {D_T:.3e} m²/s")
print(f"Rotational diffusion coefficient     D_R = {D_R:.3e} rad²/s")
print(f"Persistence length  l_p = v/D_R     = {v/D_R:.3e} m")
print(f"Persistence time    τ_R = 1/D_R     = {1/D_R:.3f} s")
print(f"Simulation duration  t  = N·dt      = {N*dt:.1f} s")

## 2 — Single-particle trajectory

In [ ]:
def simulate_abp(N, dt, v, W, D_T, D_R, x0=0.0, y0=0.0, theta0=0.0, seed=None):
    """Simulate one active Brownian particle trajectory.

    Returns arrays x, y, theta of length N+1.
    """
    rng = np.random.default_rng(seed)

    x     = np.empty(N + 1)
    y     = np.empty(N + 1)
    theta = np.empty(N + 1)

    x[0], y[0], theta[0] = x0, y0, theta0

    sqrt2DT = np.sqrt(2 * D_T * dt)
    sqrt2DR = np.sqrt(2 * D_R * dt)

    # pre-draw all noise at once for speed
    xi_x   = rng.standard_normal(N)
    xi_y   = rng.standard_normal(N)
    xi_th  = rng.standard_normal(N)

    for n in range(N):
        th = theta[n]
        x[n+1]     = x[n]     + v * np.cos(th) * dt + sqrt2DT * xi_x[n]
        y[n+1]     = y[n]     + v * np.sin(th) * dt + sqrt2DT * xi_y[n]
        theta[n+1] = th       + W * dt               + sqrt2DR * xi_th[n]

    return x, y, theta


x, y, theta = simulate_abp(N, dt, v, W, D_T, D_R, seed=42)
t = np.arange(N + 1) * dt

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# trajectory coloured by time
sc = axes[0].scatter(x * 1e6, y * 1e6, c=t, cmap='plasma', s=1, linewidths=0)
axes[0].plot(x[0]  * 1e6, y[0]  * 1e6, 'go', ms=8, label='start', zorder=5)
axes[0].plot(x[-1] * 1e6, y[-1] * 1e6, 'r*', ms=10, label='end',  zorder=5)
plt.colorbar(sc, ax=axes[0], label='time [s]')
axes[0].set_xlabel('x [µm]')
axes[0].set_ylabel('y [µm]')
axes[0].set_title('Active Brownian particle trajectory')
axes[0].set_aspect('equal')
axes[0].legend()

# orientation angle over time
axes[1].plot(t, np.unwrap(theta), color='steelblue', lw=0.8)
axes[1].set_xlabel('time [s]')
axes[1].set_ylabel('θ [rad]  (unwrapped)')
axes[1].set_title('Orientation angle')

plt.tight_layout()
plt.savefig('single_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: single_trajectory.png')

## 3 — Velocity autocorrelation function (VACF)

For an active Brownian particle the orientational autocorrelation decays as
$\langle \cos[\theta(t) - \theta(0)] \rangle = e^{-D_R t}$,
so the **VACF** is $C_v(\tau) = v^2 e^{-D_R \tau}$.

In [ ]:
# estimate VACF from the single trajectory via direct averaging
vx = np.diff(x) / dt
vy = np.diff(y) / dt

max_lag = 2000   # lags to compute
lags = np.arange(max_lag)
Cv = np.array([
    np.mean(vx[:N-lag] * vx[lag:] + vy[:N-lag] * vy[lag:])
    for lag in lags
])

tau = lags * dt
Cv_theory = v**2 * np.exp(-D_R * tau)

plt.figure(figsize=(7, 4))
plt.plot(tau, Cv,         color='steelblue', lw=1,   label='simulation')
plt.plot(tau, Cv_theory,  color='tomato',    lw=1.5, ls='--', label=r'$v^2 e^{-D_R \tau}$')
plt.xlabel(r'lag $\tau$ [s]')
plt.ylabel(r'$C_v(\tau)$  [m²/s²]')
plt.title('Velocity autocorrelation function')
plt.legend()
plt.tight_layout()
plt.savefig('vacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: vacf.png')

## 4 — Mean-square displacement (MSD) — ensemble average

The theoretical MSD for an active Brownian particle is:

$$\langle r^2(\tau) \rangle = 4D_T\tau + \frac{2v^2}{D_R^2}\left[D_R\tau - 1 + e^{-D_R\tau}\right]$$

* Short times ($\tau \ll 1/D_R$): ballistic, $\langle r^2\rangle \approx v^2\tau^2$
* Long times ($\tau \gg 1/D_R$): diffusive, $\langle r^2\rangle \approx (4D_T + 2v^2/D_R)\tau$

In [ ]:
N_particles = 500   # ensemble size
N_msd       = 5_000 # steps per particle

rng_ens = np.random.default_rng(0)

# vectorised ensemble simulation
x_ens     = np.zeros(N_particles)
y_ens     = np.zeros(N_particles)
theta_ens = rng_ens.uniform(0, 2 * np.pi, N_particles)  # random initial angles

sqrt2DT = np.sqrt(2 * D_T * dt)
sqrt2DR = np.sqrt(2 * D_R * dt)

# store displacement² at every step
msd_sum = np.zeros(N_msd + 1)
msd_sum[0] = 0.0

x0_ens = x_ens.copy()
y0_ens = y_ens.copy()

for n in range(N_msd):
    xi_x   = rng_ens.standard_normal(N_particles)
    xi_y   = rng_ens.standard_normal(N_particles)
    xi_th  = rng_ens.standard_normal(N_particles)

    x_ens     += v * np.cos(theta_ens) * dt + sqrt2DT * xi_x
    y_ens     += v * np.sin(theta_ens) * dt + sqrt2DT * xi_y
    theta_ens += W * dt                     + sqrt2DR * xi_th

    dr2 = (x_ens - x0_ens)**2 + (y_ens - y0_ens)**2
    msd_sum[n + 1] = dr2.mean()

tau_msd = np.arange(N_msd + 1) * dt

# ── theoretical MSD ───────────────────────────────────────────────────────────
def msd_theory(tau, D_T, D_R, v):
    return 4*D_T*tau + (2*v**2/D_R**2) * (D_R*tau - 1 + np.exp(-D_R*tau))

msd_th = msd_theory(tau_msd, D_T, D_R, v)

# ── ballistic & diffusive asymptotes ─────────────────────────────────────────
msd_ballistic = v**2 * tau_msd**2
D_eff = D_T + v**2 / (2 * D_R)
msd_diffusive = 4 * D_eff * tau_msd

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(tau_msd[1:], msd_sum[1:] * 1e12,    color='steelblue', lw=1.5, label='simulation (ensemble)')
ax.loglog(tau_msd[1:], msd_th[1:] * 1e12,     color='tomato',    lw=2,   ls='--', label='theory')
ax.loglog(tau_msd[1:], msd_ballistic[1:]*1e12, color='green',     lw=1,   ls=':',  label=r'ballistic $\propto\tau^2$')
ax.loglog(tau_msd[1:], msd_diffusive[1:]*1e12, color='orange',    lw=1,   ls=':',  label=r'diffusive $\propto\tau$')
ax.axvline(1/D_R, color='grey', ls='--', lw=0.8, label=r'$\tau=1/D_R$')
ax.set_xlabel(r'lag $\tau$ [s]')
ax.set_ylabel(r'MSD [µm²]')
ax.set_title(f'Mean-square displacement  ({N_particles} particles)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('msd.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: msd.png')

## 5 — Effect of chirality: Ω = 0 vs Ω ≠ 0

When the angular velocity $\Omega \neq 0$, the particle swims in circles on short timescales before diffusion randomises its orientation.

In [ ]:
cases = [
    dict(label='passive (v=0, Ω=0)',     v=0,   W=0,    color='grey'),
    dict(label='active, no chirality',    v=v,   W=0,    color='steelblue'),
    dict(label=f'chiral (Ω={W:.2f} rad/s)', v=v, W=W,  color='tomato'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, case in zip(axes, cases):
    xc, yc, _ = simulate_abp(N, dt, case['v'], case['W'], D_T, D_R, seed=7)
    sc = ax.scatter(xc * 1e6, yc * 1e6, c=np.arange(N+1)*dt,
                    cmap='viridis', s=1, linewidths=0)
    ax.plot(xc[0]*1e6,  yc[0]*1e6,  'go', ms=7, zorder=5)
    ax.plot(xc[-1]*1e6, yc[-1]*1e6, 'r*', ms=9, zorder=5)
    plt.colorbar(sc, ax=ax, label='time [s]')
    ax.set_xlabel('x [µm]')
    ax.set_ylabel('y [µm]')
    ax.set_title(case['label'])
    ax.set_aspect('equal')

plt.suptitle('Trajectories: passive vs active vs chiral', fontsize=13)
plt.tight_layout()
plt.savefig('chirality_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: chirality_comparison.png')

## 6 — Summary

| Quantity | Symbol | Value |
|---|---|---|
| Translational diffusion | $D_T$ | 2.20 × 10⁻¹³ m²/s |
| Rotational diffusion | $D_R$ | 0.165 rad²/s |
| Persistence time | $1/D_R$ | ~6.1 s |
| Effective swim diffusivity | $v^2/(2D_R)$ | 3.04 × 10⁻¹⁰ m²/s |

The simulation reproduces the **ballistic → diffusive crossover** in the MSD and the exponential decay of the VACF, both consistent with the analytical theory for active Brownian motion.